In [ ]:
Guardian_API=""

In [ ]:
import requests
import pandas as pd
import time
import re

# --- CONFIG ---
base_url = "https://content.guardianapis.com/search"
api_key = Guardian_API  # <-- replace with your key
page_size = 50  # reasonable size per request
# ----------------

# Movies/shows with metadata
movies = [
    {"title": "Atypical", "type": "tv", "year": 2017},
    {"title": "The Good Doctor", "type": "tv", "year": 2017},
    {"title": "Temple Grandin", "type": "film", "year": 2010},
    {"title": "Everything's Gonna Be Okay", "type": "tv", "year": 2020},
    {"title": "Taare Zameen Par", "type": "film", "year": 2007, "alt_titles": ["Like Stars on Earth"]},
    {"title": "Sitare Zameen Par", "type": "film", "year": 2025},
    {"title": "Front of the Class", "type": "film", "year": 2008},
    {"title": "The Tic Code", "type": "film", "year": 1998},
    {"title": "Patience", "type": "film", "year": 2017},
    {"title": "Barfi", "type": "film", "year": 2012},
    {"title": "My Name is Khan", "type": "film", "year": 2010, "alt_titles": ["MNIK"]},
    {"title": "Music", "type": "film", "year": 2021, "director": "Sia"},
    {"title": "Hichki", "type": "film", "year": 2018, "alt_titles": ["Hiccup"]},  # informal translation
    {"title": "Rain Man", "type": "film", "year": 1988},
    {"title": "Koi... Mil Gaya", "type": "film", "year": 2003, "alt_titles": ["Found Someone", "I Have Found Someone"]},
    {"title": "Extraordinary Attorney Woo", "type": "tv", "year": 2022, "alt_titles": ["Weird Lawyer Woo Young-woo"]}
]

# Neurodivergence-related terms
neuro_terms = ["autism", "autistic", "neurodivergent", "Asperger", "ADHD",
               "representation", "portrayal", "spectrum"]

# Context words that suggest movie discussion
context_words = ['film', 'movie', 'series', 'show', 'character', 'portrays',
                 'depicts', 'stars', 'performance', 'actor', 'actress',
                 'review', 'watch', 'streaming', 'episode', 'season']


def mentions_title_with_boundary(text: str, title: str, alt_titles: list = None) -> bool:
    """
    Check if title is mentioned with word boundaries to avoid false positives.
    Also checks alternate titles if provided.
    """
    lower_text = text.lower()

    # Check main title
    if len(title.split()) <= 2:
        # For short titles, use word boundaries
        pattern = r'\b' + re.escape(title.lower()) + r'\b'
        if re.search(pattern, lower_text):
            return True
    else:
        # For longer titles, substring match is safer
        if title.lower() in lower_text:
            return True

    # Check alternate titles
    if alt_titles:
        for alt in alt_titles:
            if alt.lower() in lower_text:
                return True

    return False


def has_context_words(text: str) -> bool:
    """Check if text contains movie-related context words."""
    lower_text = text.lower()
    return any(word in lower_text for word in context_words)


def count_mentions(text: str, title: str, alt_titles: list = None) -> int:
    """Count how many times a movie is mentioned in text."""
    lower_text = text.lower()
    count = 0

    # Count main title
    if len(title.split()) <= 2:
        pattern = r'\b' + re.escape(title.lower()) + r'\b'
        count += len(re.findall(pattern, lower_text))
    else:
        # For multi-word titles, count overlapping occurrences
        start = 0
        while True:
            pos = lower_text.find(title.lower(), start)
            if pos == -1:
                break
            count += 1
            start = pos + 1

    # Count alternate titles
    if alt_titles:
        for alt in alt_titles:
            count += lower_text.count(alt.lower())

    return count


def search_movie(movie_info: dict, api_key: str) -> list:
    """
    Search for articles about a specific movie with neurodivergence context.
    Returns list of article dictionaries.
    """
    title = movie_info["title"]
    movie_type = movie_info.get("type", "film")
    alt_titles = movie_info.get("alt_titles", [])
    director = movie_info.get("director", "")

    # Build search query
    neuro_query = " OR ".join(neuro_terms)

    # Special case for "Music" - add director to avoid generic matches
    if title == "Music" and director:
        query = f'"{title}" AND "{director}" AND ({neuro_query})'
    else:
        query = f'"{title}" AND ({neuro_query})'

    # Determine section based on type
    section = "tv-and-radio" if movie_type == "tv" else "film"

    articles = []
    seen_ids = set()
    page = 1

    print(f"Searching for: {title}...")

    while page <= 3:  # Limit to 3 pages per movie to avoid excessive API calls
        params = {
            "api-key": api_key,
            "q": query,
            "type": "article",
            "section": f"{section}|film|tv-and-radio",  # Cast wider net
            "show-fields": "headline,bodyText,standfirst,byline",
            "show-tags": "tone",
            "page-size": page_size,
            "page": page,
        }

        try:
            resp = requests.get(base_url, params=params)
            if resp.status_code != 200:
                print(f"  Warning: API error {resp.status_code} for {title}")
                break

            data = resp.json().get("response", {})
            results = data.get("results", [])

            if not results:
                break

            for r in results:
                aid = r.get("id")
                if aid in seen_ids:
                    continue
                seen_ids.add(aid)

                headline = r.get("fields", {}).get("headline", "")
                body = r.get("fields", {}).get("bodyText", "")
                standfirst = r.get("fields", {}).get("standfirst", "")
                byline = r.get("fields", {}).get("byline", "")

                combined = f"{headline} {standfirst} {body}".strip()

                # Verify the movie is actually mentioned (Guardian search isn't perfect)
                if not mentions_title_with_boundary(combined, title, alt_titles):
                    continue

                # Check for movie context
                if not has_context_words(combined):
                    continue

                # Count mentions
                mention_count = count_mentions(combined, title, alt_titles)

                # Get tone tags (e.g., "review", "feature")
                tags = [tag.get("webTitle", "") for tag in r.get("tags", [])]

                articles.append({
                    "article_id": aid,
                    "movie_title": title,
                    "movie_type": movie_type,
                    "headline": headline,
                    "byline": byline,
                    "standfirst": standfirst,
                    "url": r.get("webUrl"),
                    "review": body,
                    "mention_count": mention_count,
                    "tags": ", ".join(tags),
                    "in_headline": title.lower() in headline.lower(),
                    "source": "Guardian"
                })

            # Check if more pages available
            current_page = data.get("currentPage", page)
            total_pages = data.get("pages", page)
            if current_page >= total_pages:
                break

            page += 1
            time.sleep(0.3)  # Polite rate limiting

        except Exception as e:
            print(f"  Error searching {title}: {e}")
            break

    print(f"  Found {len(articles)} articles for {title}")
    return articles


# --- Main Execution ---
print("Starting movie search...\n")
all_articles = []

for movie in movies:
    articles = search_movie(movie, api_key)
    all_articles.extend(articles)
    time.sleep(0.5)  # Pause between movies

# --- Build DataFrame ---
if not all_articles:
    print("\nNo articles found. Check your API key and search parameters.")
else:
    df = pd.DataFrame(all_articles)

    # Filter for quality: must be in headline OR mentioned multiple times
    df_filtered = df[
        (df["in_headline"] == True) |
        (df["mention_count"] >= 2)
    ].copy()

    # Sort by movie and mention count
    df_filtered = df_filtered.sort_values(
        ["movie_title", "mention_count"],
        ascending=[True, False]
    )

    # Save both full and filtered results
    df.to_csv("guardian_neurodivergence_all_articles.csv", index=False)
    df_filtered.to_csv("guardian_neurodivergence_filtered_articles.csv", index=False)

    # Display summary
    print(f"\n{'='*60}")
    print(f"SEARCH COMPLETE")
    print(f"{'='*60}")
    print(f"Total articles found: {len(df)}")
    print(f"High-quality articles: {len(df_filtered)}")
    print(f"\nArticles by movie:")
    print(df_filtered.groupby("movie_title").size().to_string())
    print(f"\n{'='*60}")
    print("\nSample of filtered results:")
    print(df_filtered[["movie_title", "headline", "mention_count", "in_headline"]].head(15).to_string())
    print(f"\n{'='*60}")
    print("Files saved:")
    print("  - guardian_neurodivergence_all_articles.csv (all results)")
    print("  - guardian_neurodivergence_filtered_articles.csv (high quality)")

Starting movie search...

Searching for: Atypical...
  Found 35 articles for Atypical
Searching for: The Good Doctor...
  Found 15 articles for The Good Doctor
Searching for: Temple Grandin...
  Found 10 articles for Temple Grandin
Searching for: Everything's Gonna Be Okay...
  Found 0 articles for Everything's Gonna Be Okay
Searching for: Taare Zameen Par...
  Found 0 articles for Taare Zameen Par
Searching for: Sitare Zameen Par...
  Found 0 articles for Sitare Zameen Par
Searching for: Front of the Class...
  Found 2 articles for Front of the Class
Searching for: The Tic Code...
  Found 0 articles for The Tic Code
Searching for: Patience...
  Found 78 articles for Patience
Searching for: Barfi...
  Found 2 articles for Barfi
Searching for: My Name is Khan...
  Found 4 articles for My Name is Khan
Searching for: Music...
  Found 9 articles for Music
Searching for: Hichki...
  Found 0 articles for Hichki
Searching for: Rain Man...
  Found 42 articles for Rain Man
Searching for: Koi...

In [ ]:
from google.colab import files

files.download("guardian_neurodivergence_all_articles.csv")
files.download("guardian_neurodivergence_filtered_articles.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **MORE TWEAKINGS FOR BETTER QUALITY AND QUANTITY**

In [ ]:
import requests
import pandas as pd
import time
import re
from difflib import SequenceMatcher
import os

# --- CONFIG ---
base_url = "https://content.guardianapis.com/search"
api_key = Guardian_API
page_size = 50
max_pages = 3
rate_delay = 0.3
# ----------------

movies = [
    {"title": "Atypical", "type": "tv", "year": 2017},
    {"title": "The Good Doctor", "type": "tv", "year": 2017},
    {"title": "Temple Grandin", "type": "film", "year": 2010},
    {"title": "Everything's Gonna Be Okay", "type": "tv", "year": 2020},
    {"title": "Taare Zameen Par", "type": "film", "year": 2007, "alt_titles": ["Like Stars on Earth"]},
    {"title": "Sitare Zameen Par", "type": "film", "year": 2025},
    {"title": "Front of the Class", "type": "film", "year": 2008},
    {"title": "The Tic Code", "type": "film", "year": 1998},
    {"title": "Patience", "type": "film", "year": 2017},
    {"title": "Barfi", "type": "film", "year": 2012},
    {"title": "My Name is Khan", "type": "film", "year": 2010, "alt_titles": ["MNIK"]},
    {"title": "Music", "type": "film", "year": 2021, "director": "Sia"},
    {"title": "Hichki", "type": "film", "year": 2018, "alt_titles": ["Hiccup"]},
    {"title": "Rain Man", "type": "film", "year": 1988},
    {"title": "Koi... Mil Gaya", "type": "film", "year": 2003, "alt_titles": ["Found Someone", "I Have Found Someone"]},
    {"title": "Extraordinary Attorney Woo", "type": "tv", "year": 2022, "alt_titles": ["Weird Lawyer Woo Young-woo"]}
]

neuro_terms = [
    "autism", "autistic", "Asperger", "ADHD", "neurodivergent",
    "learning disability", "developmental disorder", "mental health",
    "special needs", "savant", "neurotypical", "representation",
    "portrayal", "inclusion", "diversity"
]

context_words = [
    "film", "movie", "series", "show", "character", "portrays", "depicts",
    "stars", "performance", "actor", "actress", "review", "watch",
    "streaming", "episode", "season", "drama", "director"
]


def mentions_title_with_boundary(text: str, title: str, alt_titles: list = None) -> bool:
    lower_text = text.lower()
    # main title
    if len(title.split()) <= 2:
        pattern = r'\b' + re.escape(title.lower()) + r'\b'
        if re.search(pattern, lower_text):
            return True
    else:
        if title.lower() in lower_text:
            return True

    # alt titles
    if alt_titles:
        for alt in alt_titles:
            if alt.lower() in lower_text:
                return True
    return False


def fuzzy_title_match(title: str, text: str, threshold: float = 0.8) -> bool:
    """Fallback fuzzy check for stylized title mentions"""
    lower_text = text.lower()
    for phrase in re.findall(r"[a-z0-9\s\-']{4,}", lower_text):
        if SequenceMatcher(None, phrase.strip(), title.lower()).ratio() > threshold:
            return True
    return False


def has_context_words(text: str) -> int:
    """Return number of context words found (context score)."""
    lower_text = text.lower()
    return sum(word in lower_text for word in context_words)


def count_mentions(text: str, title: str, alt_titles: list = None) -> int:
    lower_text = text.lower()
    count = 0
    if len(title.split()) <= 2:
        pattern = r'\b' + re.escape(title.lower()) + r'\b'
        count += len(re.findall(pattern, lower_text))
    else:
        start = 0
        while True:
            pos = lower_text.find(title.lower(), start)
            if pos == -1:
                break
            count += 1
            start = pos + 1

    if alt_titles:
        for alt in alt_titles:
            count += lower_text.count(alt.lower())
    return count


def matched_neuro_terms(text: str) -> list:
    lower_text = text.lower()
    return [term for term in neuro_terms if term in lower_text]


def search_movie(movie_info: dict, api_key: str) -> list:
    title = movie_info["title"]
    movie_type = movie_info.get("type", "film")
    alt_titles = movie_info.get("alt_titles", [])
    director = movie_info.get("director", "")
    year = movie_info.get("year", None)

    # Build multiple queries
    neuro_query = " OR ".join(neuro_terms)
    queries = [
        f'"{title}" AND ({neuro_query})',
        f'"{title}" AND (review OR film OR movie OR "tv series")',
        f'"{title}" AND (character OR representation OR "main character")'
    ]
    if title == "Music" and director:
        queries.insert(0, f'"{title}" AND "{director}" AND ({neuro_query})')

    articles = []
    seen_urls = set()

    print(f"Searching for: {title}...")

    for query in queries:
        page = 1
        while page <= max_pages:
            params = {
                "api-key": api_key,
                "q": query,
                "type": "article",
                "show-fields": "headline,bodyText,standfirst,byline",
                "show-tags": "tone",
                "page-size": page_size,
                "page": page,
                "from-date": f"{year-1}-01-01" if year else "2000-01-01",
                "to-date": f"{year+1}-12-31" if year else "2025-12-31"
            }

            try:
                resp = requests.get(base_url, params=params)
                if resp.status_code != 200:
                    print(f"  API error {resp.status_code} for {title}")
                    break

                data = resp.json().get("response", {})
                results = data.get("results", [])
                if not results:
                    break

                for r in results:
                    url = r.get("webUrl")
                    if url in seen_urls:
                        continue
                    seen_urls.add(url)

                    headline = r.get("fields", {}).get("headline", "")
                    body = r.get("fields", {}).get("bodyText", "")
                    standfirst = r.get("fields", {}).get("standfirst", "")
                    byline = r.get("fields", {}).get("byline", "")
                    section = r.get("sectionName", "")
                    date = r.get("webPublicationDate", "")
                    tags = [tag.get("webTitle", "") for tag in r.get("tags", [])]
                    combined = f"{headline} {standfirst} {body}".strip()

                    # Verify title mention
                    if not (mentions_title_with_boundary(combined, title, alt_titles)
                            or fuzzy_title_match(title, combined)):
                        continue

                    context_score = has_context_words(combined)
                    if context_score == 0:
                        continue

                    mention_count = count_mentions(combined, title, alt_titles)
                    neuro_hits = matched_neuro_terms(combined)

                    articles.append({
                        "movie_title": title,
                        "movie_type": movie_type,
                        "headline": headline,
                        "byline": byline,
                        "url": url,
                        "publication_date": date,
                        "section": section,
                        "mention_count": mention_count,
                        "context_score": context_score,
                        "matched_neuro_terms": ", ".join(neuro_hits),
                        "tags": ", ".join(tags),
                        "in_headline": title.lower() in headline.lower(),
                        "source": "Guardian",
                        "article_text": combined
                    })

                if data.get("currentPage", page) >= data.get("pages", page):
                    break
                page += 1
                time.sleep(rate_delay)
            except Exception as e:
                print(f"  Error searching {title}: {e}")
                break

        time.sleep(rate_delay)

    print(f"  Found {len(articles)} articles for {title}")
    return articles


# --- MAIN EXECUTION ---
print("Starting enhanced Guardian search...\n")
all_articles = []

for movie in movies:
    results = search_movie(movie, api_key)
    all_articles.extend(results)
    time.sleep(0.5)

if not all_articles:
    print("\nNo articles found. Check API key or network connection.")
else:
    df = pd.DataFrame(all_articles)
    df["score"] = (
        df["mention_count"] * 2 +
        df["in_headline"].astype(int) * 3 +
        df["context_score"] +
        df["matched_neuro_terms"].apply(lambda x: len(x.split(", ")) if x else 0)
    )

    df_sorted = df.sort_values(["score", "mention_count"], ascending=[False, False])

    df.to_csv("guardian_neurodivergence_all_v2.csv", index=False)
    df_sorted.to_csv("guardian_neurodivergence_top_v2.csv", index=False)

    print(f"\n{'='*60}")
    print("SEARCH COMPLETE")
    print(f"{'='*60}")
    print(f"Total articles found: {len(df)}")
    print(f"Movies covered: {df['movie_title'].nunique()}")
    print(f"\nTop high-quality reviews:\n")
    print(df_sorted[["movie_title", "headline", "score", "section", "in_headline"]].head(15).to_string(index=False))
    print(f"\n{'='*60}")
    print("Files saved:")
    print("  - guardian_neurodivergence_all_v2.csv (all results)")
    print("  - guardian_neurodivergence_top_v2.csv (ranked high quality)")


Starting enhanced Guardian search...

Searching for: Atypical...
  Found 118 articles for Atypical
Searching for: The Good Doctor...
  Found 25 articles for The Good Doctor
Searching for: Temple Grandin...
  Found 8 articles for Temple Grandin
Searching for: Everything's Gonna Be Okay...
  Found 1 articles for Everything's Gonna Be Okay
Searching for: Taare Zameen Par...
  Found 1 articles for Taare Zameen Par
Searching for: Sitare Zameen Par...
  Found 0 articles for Sitare Zameen Par
Searching for: Front of the Class...
  Found 14 articles for Front of the Class
Searching for: The Tic Code...
  Found 0 articles for The Tic Code
Searching for: Patience...
  Found 351 articles for Patience
Searching for: Barfi...
  Found 3 articles for Barfi
Searching for: My Name is Khan...
  Found 23 articles for My Name is Khan
Searching for: Music...
  Found 385 articles for Music
Searching for: Hichki...
  Found 1 articles for Hichki
Searching for: Rain Man...
  Found 0 articles for Rain Man
Searc

In [ ]:
from google.colab import files

files.download("guardian_neurodivergence_all_v2.csv")
files.download("guardian_neurodivergence_top_v2.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>